# AGEMR B3：固定 BP target，改变 BMI source 年龄构成

先回答一个具体问题：**在同一 BP 目标样本、同一工具 SNP 和同一来源样本量下，改变 BMI 来源的年龄分布，naive two-sample MR 的结果是否系统变化？**

这是 slides 第 22 页 B3 的 baseline：IVW、weighted median、MR-Egger；同时跑 BMI→SBP 和 BMI→DBP。保留现有非线性 age spline、sex、QPC nuisance 的完整列空间。这里估计的是各样本中的 **pooled SNP 主效应**，不把 B1 interaction 模型的 ADD 行当作 SNP–BMI 主效应。

只读取现有 workspace 持久文件和 PGEN；不下载 controlled data，也不覆盖 B1/旧结果。无需 Spark collect、额外 R 包或 controlled-data 网络请求。首次运行会新建独立输出目录。

**预设设计：** 固定 IID hash 分成 discovery 35%、BP target 20%、BMI source pool 45%，同一人跨表型只能属于同一角色。默认独立 discovery 最多 80,000 人、固定 target 最多 30,000 人、每个 source 最多 30,000 人，3 次配对重复。可用人数不足时明确显示实际人数；source 按所有情境共同支持度一次性降低 N，以后固定。

IV 在独立 discovery 中按 BMI P<5×10⁻⁸、F≥10、MAF≥0.01、MAC≥20 选一次，并用 **AoU discovery 自己的 LD** 做 10 Mb、r²=0.001 clumping。原候选面板的来源不等于已核实的筛选历史；这里重新估计 discovery P 和 LD。分析仍只覆盖现有候选面板，不称为全基因组 MR。若没有合格 IV，保存诊断后停止，不自动放宽 P。

没有外部权重或独立的既有 discovery 结果，因此这次先用三角色分割。现有 B1 full-sample 显著性不会参与新 IV 选择。若提供已有无关个体 keep，可设置 `UNRELATED_KEEP`；按 IID 分开不能自动排除跨角色亲缘关系。

In [ ]:
from pathlib import Path
import json, hashlib, uuid, subprocess, shutil, warnings, gc
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats, linalg, sparse
import matplotlib.pyplot as plt
from IPython.display import display

# Only this configuration cell normally needs editing.
LINEAR_RUN = None
B1_ROOT = Path.home()/"workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE = None  # e.g. str(Path.home()/"plink2_env/bin/plink2")
UNRELATED_KEEP = None  # Optional existing IID or FID=0 keep; no new kinship inference.
SEED = 20261028
N_DISCOVERY_REQUESTED = 80000
N_TARGET_REQUESTED = 30000
N_SOURCE_REQUESTED = 30000
N_REPEATS = 3
N_BOOT = 2000
DISCOVERY_P = 5e-8
DISCOVERY_F_MIN = 10
MIN_MAF = .01
MIN_MAC = 20
CLUMP_R2 = .001
CLUMP_KB = 10000
MAX_FIXED_IV = 100  # Memory guard; do not select top SNPs to satisfy this limit.
MIN_ROLE_N = 100
THREADS = 2
MEMORY_MB = 1024
AGE_BIN_YEARS = 5
SCENARIOS = [("matched", 0.), ("younger_moderate", -.5),
             ("younger_strong", -1.), ("older_moderate", .5), ("older_strong", 1.)]
OUTCOMES = ("SBP", "DBP")
METHODS = ("IVW", "weighted_median", "MR_Egger")
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
CONTRAST_FAMILY = len(OUTCOMES)*(len(SCENARIOS)-1)*len(METHODS)

def require(ok, message):
    if not ok: raise RuntimeError(message)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024), b""): h.update(b)
    return h.hexdigest()

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str))

def iid_table(path):
    x = pd.read_csv(path, sep=r"\s+", dtype=str, keep_default_na=False)
    x = x.rename(columns={"#IID":"IID", "#FID":"FID"})
    require("IID" in x and x.IID.is_unique, f"Missing/duplicate IID: {path}")
    if "FID" in x:
        require(x.FID.eq("0").all(), "This workflow requires IID-only or FID=0 inputs")
        x = x.drop(columns="FID")
    return x.set_index("IID")

def stable_rng(label):
    value = int.from_bytes(hashlib.sha256(f"{SEED}|{label}".encode()).digest()[:8], "big")
    return np.random.default_rng(value)

def role(iid):
    u = int.from_bytes(hashlib.sha256(f"B3|{SEED}|{iid}".encode()).digest()[:8], "big")/2**64
    return "discovery" if u < .35 else ("target" if u < .55 else "source")

def run_cmd(command, prefix):
    prefix = Path(prefix); prefix.parent.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    write_json(str(prefix)+".command.json", command)
    with Path(str(prefix)+".stdout.txt").open("w") as out, Path(str(prefix)+".stderr.txt").open("w") as err:
        res = subprocess.run(command, stdout=out, stderr=err)
    if res.returncode:
        detail = Path(str(prefix)+".stderr.txt").read_text(errors="replace")[-2500:]
        raise RuntimeError(f"PLINK failed ({res.returncode}): {prefix}\n{detail}")

def nuisance_basis(C):
    X = np.column_stack([np.ones(len(C)), np.asarray(C, float)])
    Q, R, piv = linalg.qr(X, mode="economic", pivoting=True, check_finite=False)
    diag = np.abs(np.diag(R))
    rank = int((diag > max(X.shape)*np.finfo(float).eps*max(diag.max(), 1)).sum())
    return Q[:, :rank], rank, piv[:rank]

require(N_BOOT >= 200 and N_REPEATS >= 1, "Use at least 200 draws and one repeat")
if LINEAR_RUN is None:
    choices = []
    for marker in (B1_ROOT/"B1_linear_age").glob("*/RUN_COMPLETE.json"):
        m = json.loads(marker.read_text())
        if m.get("status") == "complete" and all((marker.parent/"inputs"/f"{t}.meta.json").is_file() for t in ("BMI",)+OUTCOMES):
            choices.append((m.get("finished_utc", ""), marker.parent))
    require(bool(choices), "No completed B1 linear run; set LINEAR_RUN to the existing run")
    LINEAR_RUN = max(choices, key=lambda z:z[0])[1]
else: LINEAR_RUN = Path(LINEAR_RUN).expanduser()
completion = json.loads((LINEAR_RUN/"RUN_COMPLETE.json").read_text())
require(completion.get("status") == "complete", "Selected run is incomplete")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT = LINEAR_RUN/"B3_MR_age_mismatch"/stamp
for name in ("qc", "design", "discovery", "genotypes", "associations", "MR", "figures"):
    (OUT/name).mkdir(parents=True, exist_ok=True)

PLINK = None
probe_log = []
for candidate in [PLINK_EXECUTABLE, str(Path.home()/"plink2_env/bin/plink2"), shutil.which("plink2")]:
    if not candidate: continue
    try:
        res = subprocess.run([str(candidate), "--version"], capture_output=True, text=True, timeout=30)
        msg = (res.stdout+res.stderr).strip()
        probe_log.append({"path":str(candidate), "returncode":res.returncode, "message":msg})
        if res.returncode == 0 and "PLINK" in msg:
            PLINK = str(candidate); PLINK_VERSION = msg; break
    except (OSError, subprocess.TimeoutExpired) as e:
        probe_log.append({"path":str(candidate), "error":str(e)})
write_json(OUT/"qc/plink_discovery.json", probe_log)
require(PLINK is not None, "No executable PLINK2; set PLINK_EXECUTABLE to the working HOME installation, not a non-executable workspace mount")

records = json.loads((LINEAR_RUN/"qc/input_files.json").read_text())
pfiles = {int(Path(r["path"]).stem.replace("BMI_chr", "")):Path(r["path"]).with_suffix("")
          for r in records if Path(r["path"]).suffix == ".pgen"}
for record in records:
    p = Path(record["path"])
    if p.suffix in (".pgen", ".pvar", ".psam"):
        st = p.stat()
        require(st.st_size == record["bytes"] and st.st_mtime_ns == record["mtime_ns"], f"Genotype input changed: {p}")
panel = pd.read_csv(LINEAR_RUN/"qc/variant_panel.tsv", sep="\t", dtype={"#CHROM":str})
panel["#CHROM"] = panel["#CHROM"].str.replace(r"^chr", "", regex=True)
require(panel.ID.is_unique and not panel.duplicated(KEY).any(), "Ambiguous panel identifiers")
require(panel["#CHROM"].astype(int).between(1,22).all() and not panel.ALT.str.contains(",").any(), "Autosomal biallelic panel required")

pheno, covar, age, input_audit = {}, {}, {}, []
unrelated = None if UNRELATED_KEEP is None else set(iid_table(Path(UNRELATED_KEEP)).index)
for trait in ("BMI",)+OUTCOMES:
    pp = LINEAR_RUN/"inputs"/f"{trait}.pheno.tsv"
    cp = LINEAR_RUN/"inputs"/f"{trait}.covar.tsv"
    mp = LINEAR_RUN/"inputs"/f"{trait}.meta.json"
    meta = json.loads(mp.read_text())
    require(sha(pp) == meta["pheno_sha256"] and sha(cp) == meta["covar_sha256"], f"{trait}: saved input changed")
    ph = iid_table(pp)[[trait]].apply(pd.to_numeric, errors="raise").astype(float)
    cv = iid_table(cp).loc[ph.index].apply(pd.to_numeric, errors="raise").astype(float)
    if unrelated is not None: ph = ph.loc[ph.index.isin(unrelated)]; cv = cv.loc[ph.index]
    require(np.isfinite(ph.to_numpy()).all() and np.isfinite(cv.to_numpy()).all(), f"Missing {trait} input")
    require("AGE10" in cv and meta["age_unit_years"] == 10, "Need saved AGE10 and age center metadata")
    pheno[trait], covar[trait] = ph[trait], cv
    age[trait] = cv.AGE10*float(meta["age_unit_years"])+float(meta["age_center_years"])
    input_audit.append({"trait":trait, "N":len(ph), "pheno_sha256":sha(pp), "covar_sha256":sha(cp), "meta_sha256":sha(mp)})
pd.DataFrame(input_audit).to_csv(OUT/"qc/input_hashes.tsv", sep="\t", index=False)
print("Existing run:", LINEAR_RUN)
print("New B3 output:", OUT)
print(PLINK_VERSION)

## 1. 固定目标和样本量，制造可核查的年龄差异

SBP/DBP 使用同一组可用目标 IID，BP 目标及其 SNP–BP 结果始终固定。BMI source 的年龄来自 BMI 测量年龄，BP target 来自 BP 测量年龄；从保存的 AGE10 与中心化元数据准确恢复。

按 5 岁年龄箱匹配 target 分布。五个 source 情境为：matched、moderate/strong younger、moderate/strong older。箱概率为

\[q_\kappa(b)\propto \pi_{target}(b)\exp\{\kappa(a_b-\bar a_T)/s_T\},\qquad \kappa=0,\pm0.5,\pm1.\]

每组无放回抽样，各组 N 相等；同一次重复共享箱内人员随机排序，使比较配对。不同组及重复之间允许重用 source 人员，其重叠将进入协方差计算。matched 只保证箱频率近似一致，仍需查看真实均值、SD、KS statistic 和 Wasserstein 距离。

输出的 age-bin reweight ESS 仅是已抽到年龄箱上的 overlap/positivity 诊断；还会报告 target 落在未抽到箱中的概率质量。若该质量不可忽略，当前 source 无法覆盖完整 target，需调整设计。本 baseline 的 GWAS/MR **没有使用 IPW**。ESS 不能替代回归标准误或当作新的实际样本数。

In [ ]:
all_ids = pd.Index(sorted(set().union(*(set(y.index) for y in pheno.values()))), name="IID")
roles = pd.Series([role(i) for i in all_ids], index=all_ids, name="role")
roles.to_csv(OUT/"design/individual_roles.tsv", sep="\t", index_label="#IID")
bp_ids = pheno["SBP"].index.intersection(pheno["DBP"].index).sort_values()
require(np.allclose(age["SBP"].loc[bp_ids], age["DBP"].loc[bp_ids], atol=1e-7, rtol=0), "SBP/DBP ages differ; use separate target designs")
discovery_pool = pheno["BMI"].index[roles.loc[pheno["BMI"].index].eq("discovery")].sort_values()
target_pool = bp_ids[roles.loc[bp_ids].eq("target")]
source_pool = pheno["BMI"].index[roles.loc[pheno["BMI"].index].eq("source")].sort_values()
require(min(len(discovery_pool), len(target_pool), len(source_pool)) >= MIN_ROLE_N, "At least one role pool too small")
discovery_ids = pd.Index(stable_rng("discovery").choice(discovery_pool, min(N_DISCOVERY_REQUESTED, len(discovery_pool)), replace=False)).sort_values()
target_ids = pd.Index(stable_rng("target").choice(target_pool, min(N_TARGET_REQUESTED, len(target_pool)), replace=False)).sort_values()
require(not set(discovery_ids)&set(target_ids) and not set(discovery_ids)&set(source_pool) and not set(target_ids)&set(source_pool), "Role overlap")
for name, ids in [("discovery", discovery_ids), ("target", target_ids), ("source_pool", source_pool)]:
    pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{name}.keep.tsv", sep="\t", index=False)

target_age = age["SBP"].loc[target_ids].to_numpy()
source_age = age["BMI"].loc[source_pool].to_numpy()
require(np.std(target_age, ddof=1) > 0, "No target age variation")
age_min = np.floor(min(target_age.min(), source_age.min())/AGE_BIN_YEARS)*AGE_BIN_YEARS
age_max = np.floor(max(target_age.max(), source_age.max())/AGE_BIN_YEARS)*AGE_BIN_YEARS+AGE_BIN_YEARS
edges = np.arange(age_min, age_max+AGE_BIN_YEARS*.5, AGE_BIN_YEARS)
target_bin = np.searchsorted(edges, target_age, side="right")-1
source_bin = np.searchsorted(edges, source_age, side="right")-1
nbins = len(edges)-1
target_count = np.bincount(target_bin, minlength=nbins)
pool_count = np.bincount(source_bin, minlength=nbins)
pi = target_count/len(target_ids)
active = pi > 0
require((pool_count[active] > 0).all(), "Target age support missing in source pool; inspect design before changing bins")
centers = (edges[:-1]+edges[1:])/2
zcenters = (centers-target_age.mean())/target_age.std(ddof=1)
probabilities = {}
for name, tilt in SCENARIOS:
    q = pi*np.exp(tilt*zcenters)
    probabilities[name] = q/q.sum()
capacity = min(float(np.min(pool_count[active]/q[active])) for q in probabilities.values())
N_SOURCE = min(N_SOURCE_REQUESTED, int(.9*capacity))
require(N_SOURCE >= MIN_ROLE_N, "Insufficient shared age support for equal-sized sources")
if N_SOURCE < N_SOURCE_REQUESTED:
    warnings.warn(f"Common source N reduced once to {N_SOURCE} for age-bin capacity; fixed across all scenarios/repeats")

def quotas(q, n):
    v = n*np.asarray(q); a = np.floor(v).astype(int)
    extra = n-a.sum()
    if extra: a[np.argsort(-(v-a), kind="stable")[:extra]] += 1
    require(a.sum() == n and (a <= pool_count).all(), "Age-bin quota exceeds source capacity")
    return a

quota_rows, sample_rows, source_sets = [], [], []
for rep in range(N_REPEATS):
    # Common within-bin rankings make matched/mismatched comparisons paired.
    rankings = {b:stable_rng(f"source|{rep}|{b}").permutation(source_pool[source_bin == b]) for b in range(nbins)}
    for name, tilt in SCENARIOS:
        q = probabilities[name]; a = quotas(q, N_SOURCE)
        ids = pd.Index(np.concatenate([rankings[b][:a[b]] for b in range(nbins)])).sort_values()
        require(ids.is_unique and len(ids) == N_SOURCE and not set(ids)&set(target_ids), "Invalid source sample")
        sid = f"r{rep+1:02d}_{name}"
        source_sets.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "ids":ids})
        pd.DataFrame({"#IID":ids}).to_csv(OUT/"design"/f"{sid}.keep.tsv", sep="\t", index=False)
        observed = age["BMI"].loc[ids].to_numpy()
        bindex = np.searchsorted(edges, observed, side="right")-1
        realized_q = a/N_SOURCE
        diagnostic_weights = pi[bindex]/realized_q[bindex]
        ess = diagnostic_weights.sum()**2/(diagnostic_weights@diagnostic_weights)
        uncovered_mass = float(pi[(pi>0)&(a==0)].sum())
        sample_rows.append({"sid":sid, "repeat":rep+1, "scenario":name, "tilt":tilt, "N_source":len(ids),
                            "age_mean":observed.mean(), "age_SD":observed.std(ddof=1),
                            "target_age_mean":target_age.mean(), "age_mean_gap":observed.mean()-target_age.mean(),
                            "Wasserstein_years":stats.wasserstein_distance(observed,target_age),
                            "KS_statistic":stats.ks_2samp(observed,target_age).statistic,
                            "target_probability_in_unsampled_bins":uncovered_mass,
                            "age_bin_reweight_ESS_on_observed_support_only":ess})
        for b in range(nbins):
            quota_rows.append({"sid":sid, "bin_lower":edges[b], "bin_upper":edges[b+1],
                               "source_pool_count":pool_count[b], "target_count":target_count[b],
                               "target_probability":pi[b], "intended_source_probability":q[b], "N_sampled":a[b]})
sample_qc = pd.DataFrame(sample_rows)
pd.DataFrame(quota_rows).to_csv(OUT/"design/age_bin_quotas.tsv", sep="\t", index=False)
sample_qc.to_csv(OUT/"design/age_mixture_QC.tsv", sep="\t", index=False)
S = len(source_sets)
overlap = np.array([[len(set(a["ids"])&set(b["ids"])) for b in source_sets] for a in source_sets])
pd.DataFrame(overlap,index=[s["sid"] for s in source_sets],columns=[s["sid"] for s in source_sets]).to_csv(OUT/"design/source_overlap_counts.tsv",sep="\t")
design = {"seed":SEED, "N_discovery":len(discovery_ids), "N_target":len(target_ids), "N_source_each":N_SOURCE,
          "N_source_pool":len(source_pool), "repeats":N_REPEATS, "scenarios":SCENARIOS, "age_bin_years":AGE_BIN_YEARS,
          "discovery_P":DISCOVERY_P, "discovery_F_min":DISCOVERY_F_MIN, "clump_r2":CLUMP_R2, "clump_kb":CLUMP_KB,
          "N_boot":N_BOOT, "contrast_family":CONTRAST_FAMILY, "unrelated_keep":UNRELATED_KEEP,
          "nuisance":"saved full age spline + sex + QPC span", "target_age_mean":target_age.mean()}
write_json(OUT/"design/design.json", design)
display(pd.DataFrame([{k:design[k] for k in ("N_discovery","N_target","N_source_each","N_source_pool","repeats")}]))
display(sample_qc)

## 2. 独立 discovery pooled GWAS 和 AoU LD clumping

这里运行无 interaction 的 `--glm hide-covar`。协变量仅因当前样本中的精确线性依赖而降秩，再标准化，保留原 nuisance span。`--no-input-missing-phenotype` 避免把数值 -9 自动解释为缺失。

PLINK2 clump 默认读取 TEST=ADD；这里显式使用 `--clump-test ADD` 和 `--clump-unphased`，与 additive dosage regression 的未定相 genotype 相关性对应。输入是本次 discovery 的 P，LD 也仅在 discovery 角色估计；不会用 target outcome 选 IV。[PLINK2 官方 clumping 文档](https://www.cog-genomics.org/plink/2.0/postproc)。

工具集合固定后，后续所有情境保留同一集合，不按各 source 的 P/F 再筛选。这样先隔离系数/年龄构成变化；自适应 IV-selection stability、TLMR 和 transport 方法属于后续另一项比较。少于 3 个工具时，仅 IVW 可用。

In [ ]:
# Select IVs once, in the independent BMI discovery role; never reselect by source P/F.
pheno["BMI"].loc[discovery_ids].to_csv(OUT/"discovery/BMI.pheno.tsv", sep="\t", index_label="#IID", header=True, float_format="%.17g")
D = covar["BMI"].loc[discovery_ids]
_, rankD, pivD = nuisance_basis(D)
# The intercept is always part of this span; obtain a subset of original covariates
# independent of it. Standardization only changes coordinates, not adjustment.
keep_columns = []
Xcur = np.ones((len(D),1)); current_rank = 1
for col in D:
    v = D[col].to_numpy()
    if np.std(v) <= 1e-14: continue
    trial = np.column_stack([Xcur, (v-v.mean())/v.std(ddof=1)])
    if np.linalg.matrix_rank(trial) > current_rank:
        keep_columns.append(col); Xcur = trial; current_rank += 1
require(current_rank == rankD, "Discovery nuisance span changed during rank reduction")
Dc = D[keep_columns].copy()
Dc = (Dc-Dc.mean())/Dc.std(ddof=1)
Dc.to_csv(OUT/"discovery/BMI.covar.tsv",sep="\t",index_label="#IID",float_format="%.17g")
write_json(OUT/"qc/discovery_nuisance.json", {"original_columns":list(D), "kept_columns":keep_columns,"rank_with_intercept":rankD})
discovery_parts, index_ids = [], []
for chrom in sorted(pfiles):
    print(f"B3 independent discovery GWAS and clumping: chr{chrom}", flush=True)
    prefix = OUT/"discovery"/f"chr{chrom}"
    cmd = [PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design/discovery.keep.tsv",
           "--pheno",OUT/"discovery/BMI.pheno.tsv","--pheno-name","BMI",
           "--covar",OUT/"discovery/BMI.covar.tsv","--no-input-missing-phenotype",
           "--maf",MIN_MAF,"--mac",MIN_MAC,"--glm","hide-covar",
           "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix]
    run_cmd(cmd,prefix)
    x = pd.read_csv(str(prefix)+".BMI.glm.linear",sep="\t",dtype=str,keep_default_na=False)
    require(x.TEST.eq("ADD").all() and x.ID.is_unique,"Discovery report must contain one ADD row per ID")
    for col in ("P","BETA","SE","OBS_CT"): x[col] = pd.to_numeric(x[col],errors="coerce")
    x["#CHROM"] = x["#CHROM"].str.replace(r"^chr","",regex=True)
    x["POS"] = pd.to_numeric(x.POS,errors="raise").astype(int)
    x["F_discovery_classic"] = (x.BETA/x.SE)**2
    x["eligible_discovery"] = x.ERRCODE.eq(".") & np.isfinite(x.P) & x.P.ge(0) & x.P.lt(DISCOVERY_P) & x.SE.gt(0) & np.isfinite(x.BETA) & x.F_discovery_classic.ge(DISCOVERY_F_MIN)
    discovery_parts.append(x)
    candidates = x.loc[x.eligible_discovery]
    if not len(candidates): continue
    clump_input = OUT/"discovery"/f"chr{chrom}.clump_input.tsv"
    candidates.to_csv(clump_input,sep="\t",index=False,float_format="%.17g")
    cp = OUT/"discovery"/f"chr{chrom}_clump"
    run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design/discovery.keep.tsv",
             "--clump",clump_input,"--clump-test","ADD","--clump-p1",DISCOVERY_P,
             "--clump-p2",1,"--clump-kb",CLUMP_KB,"--clump-r2",CLUMP_R2,"--clump-unphased",
             "--threads",THREADS,"--memory",MEMORY_MB,"--out",cp],cp)
    require(Path(str(cp)+".clumps").is_file(),f"No clump report for eligible chr{chrom}")
    cl = pd.read_csv(str(cp)+".clumps",sep=r"\s+",dtype=str)
    require("ID" in cl,"Unexpected PLINK2 clump schema")
    index_ids.extend(cl.ID.tolist())
discovery = pd.concat(discovery_parts,ignore_index=True)
discovery.to_csv(OUT/"discovery/pooled_BMI_GWAS.tsv",sep="\t",index=False)
require(bool(index_ids),"No eligible independent discovery IV. Discovery results saved; do not automatically relax P or reuse the source/target for selection")
iv = discovery.loc[discovery.ID.isin(index_ids)].copy()
iv = iv.sort_values(["#CHROM","POS","ID"],key=lambda s:s.astype(int) if s.name=="#CHROM" else s).reset_index(drop=True)
require(len(iv) == len(set(index_ids)) and len(iv) <= MAX_FIXED_IV,"IV count inconsistent or exceeds memory guard; inspect before rerunning")
require(set(iv.A1).issubset(set(iv.REF)|set(iv.ALT)) and ((iv.A1==iv.REF)|(iv.A1==iv.ALT)).all(),"Unknown discovery effect allele")
iv["beta_discovery_ALT"] = iv.BETA*np.where(iv.A1.eq(iv.ALT),1,-1)
iv["orientation"] = np.where(iv.beta_discovery_ALT>0,1.,-1.)
iv["BMI_increasing_allele"] = np.where(iv.orientation>0,iv.ALT,iv.REF)
iv.to_csv(OUT/"discovery/fixed_IVs.tsv",sep="\t",index=False)
K = len(iv)
if K < 3: warnings.warn(f"Only {K} fixed IV(s): IVW is available; weighted median/MR-Egger will be unavailable")
print("Fixed independent-discovery instruments:",K,flush=True)
display(iv[KEY+["BMI_increasing_allele","P","F_discovery_classic"]])

## 3. 固定 IV 的 pooled 主效应、HC1 标准误和样本重叠协方差

导出时显式计数 ALT dosage，再统一定向到 **独立 discovery 的 BMI 升高等位基因**。不会在不同 source 中重复翻转方向；MR-Egger 使用这一固定定向，若 source 出现方向反转会记录，可能不同于软件默认逐数据集重新定向的结果。

每个 SNP 分别拟合 `trait ~ SNP + saved full nuisance`，按该 SNP 的 observed genotype 删除缺失，回归不做 genotype 均值填补。使用 QR/FWL 计算，并保存 ordinary OLS 与 HC1 sandwich SE。HC1 只针对个体独立下的异方差，不校正亲缘结构或上游 phenotype 问题。

同时保留各 SNP 的个体 influence 列。重叠 source 的交叉乘积构成联合系数协方差；固定 BP target 也保留 SNP 间 covariance。每次联合重抽系数时，**所有情境共享同一 BP target draw**，source draws 保留重叠相关性。其推断是固定 IV、当前样本设计下的大样本近似；不是对 discovery 选择、家系或新 cohort 的完整 bootstrap。

In [ ]:
# Explicit ALT-count export; default PLINK A export can count REF instead.
analysis_ids = pd.Index(sorted(set(discovery_ids)|set(target_ids)|set(source_pool)))
pd.DataFrame({"#IID":analysis_ids}).to_csv(OUT/"design/genotype_export.keep.tsv",sep="\t",index=False)
G = pd.DataFrame(index=analysis_ids)
for chrom_s, part in iv.groupby("#CHROM",sort=False):
    chrom = int(chrom_s); prefix = OUT/"genotypes"/f"IV_chr{chrom}"
    extract = Path(str(prefix)+".extract.txt"); alleles = Path(str(prefix)+".ALT.tsv")
    part.ID.to_csv(extract,index=False,header=False)
    part[["ID","ALT"]].to_csv(alleles,sep="\t",index=False,header=False)
    run_cmd([PLINK,"--pfile",pfiles[chrom],"--keep",OUT/"design/genotype_export.keep.tsv",
             "--extract",extract,"--export","A","--export-allele",alleles,
             "--threads",THREADS,"--memory",MEMORY_MB,"--out",prefix],prefix)
    raw = pd.read_csv(str(prefix)+".raw",sep=r"\s+",dtype={"IID":str}).set_index("IID")
    require(raw.index.is_unique and set(raw.index)==set(analysis_ids),"Exported genotype IDs differ")
    if "FID" in raw: require(raw.FID.astype(str).eq("0").all(),"Unexpected genotype FID")
    for row in part.itertuples(index=False):
        col = f"{row.ID}_{row.ALT}"
        require(col in raw,f"ALT export allele unverified: {row.ID}")
        values = pd.to_numeric(raw.loc[analysis_ids,col],errors="raise")
        require(values.dropna().between(0,2).all(),"Invalid autosomal dosage")
        G[row.ID] = values
    del raw
G = G.loc[:,iv.ID]

def associations(y, C, dosage):
    """Marginal SNP regression adjusted for the full nuisance span; no Gxage.

    HC1 influence columns permit covariance across SNPs and overlapping samples.
    Each SNP uses its observed genotypes without mean imputation for regression.
    """
    y = np.asarray(y,float); C = np.asarray(C,float); A = np.asarray(dosage,float)
    n, k = A.shape
    IF = np.zeros((n,k)); rows = []; cache = {}
    for j in range(k):
        ok = np.isfinite(A[:,j]); nobs = int(ok.sum())
        token = np.packbits(ok).tobytes()
        if token not in cache:
            Q, rC, _ = nuisance_basis(C[ok])
            yr = y[ok]-Q@(Q.T@y[ok])
            cache[token] = (Q,rC,yr)
        Q, rC, yr = cache[token]
        g = A[ok,j]; gr = g-Q@(Q.T@g); den = float(gr@gr)
        df = nobs-rC-1
        row = {"N":nobs,"DF_RESID":df,"nuisance_rank":rC,"MAC":min(np.sum(g),2*nobs-np.sum(g))}
        if df<=5 or den<=1e-10 or nobs<MIN_ROLE_N:
            rows.append({**row,"status":"unidentifiable_or_too_small","BETA":np.nan,"SE_HC1":np.nan}); continue
        b = float(gr@yr/den); e = yr-b*gr
        influence = np.sqrt(nobs/df)*gr*e/den
        IF[ok,j] = influence
        se = float(np.linalg.norm(influence)); seclassic = float(np.sqrt((e@e/df)/den))
        rows.append({**row,"status":"ok" if se>0 and np.isfinite(se) else "invalid_SE", "BETA":b,"SE_HC1":se,
                     "SE_classic":seclassic,"P_HC1":2*stats.norm.sf(abs(b/se)) if se>0 else np.nan,
                     "P_classic":2*stats.t.sf(abs(b/seclassic),df) if seclassic>0 else np.nan,
                     "F_HC1":(b/se)**2 if se>0 else np.nan})
    return pd.DataFrame(rows), IF

# LD diagnostic includes long-distance and cross-chromosome residual correlations.
GD = G.loc[discovery_ids].to_numpy()
means = np.nanmean(GD,axis=0); GD = np.where(np.isfinite(GD),GD,means)
R = np.atleast_2d(np.corrcoef(GD,rowvar=False)) if K>1 else np.ones((1,1))
pd.DataFrame(R,index=iv.ID,columns=iv.ID).to_csv(OUT/"qc/discovery_IV_genotype_correlations.tsv",sep="\t")
max_r2 = float(np.max((R*R)[np.triu_indices(K,1)])) if K>1 else 0.
if max_r2>.01: warnings.warn(f"Residual pairwise IV r2 reaches {max_r2:.3g}; inspect LD before independent-IV MR interpretation")
write_json(OUT/"qc/IV_LD_summary.json",{"K":K,"max_pairwise_r2":max_r2,"clumping":"discovery unphased genotype LD, 10 Mb by default"})
del GD

orientation = iv.orientation.to_numpy()
target_assoc, target_cov, target_IF = {}, {}, {}
for outcome in OUTCOMES:
    print("B3 fixed target associations:",outcome,flush=True)
    a, influence = associations(pheno[outcome].loc[target_ids],covar[outcome].loc[target_ids],G.loc[target_ids])
    a = pd.concat([iv[KEY].reset_index(drop=True),a],axis=1)
    a["BETA_ALT"] = a.BETA; a["BETA"] *= orientation
    influence *= orientation[None,:]
    a.to_csv(OUT/"associations"/f"fixed_target_{outcome}.tsv",sep="\t",index=False)
    require(a.status.eq("ok").all(),f"Target {outcome} has invalid fixed IV association; inspect saved report")
    target_assoc[outcome] = a
    target_IF[outcome] = influence
    target_cov[outcome] = influence.T@influence
    np.save(OUT/"associations"/f"fixed_target_{outcome}_cov.npy",target_cov[outcome])

source_assoc, sparse_IF, strength_rows = [], [], []
for s in source_sets:
    print("B3 source associations:",s["sid"],flush=True)
    ids = s["ids"]
    a, influence = associations(pheno["BMI"].loc[ids],covar["BMI"].loc[ids],G.loc[ids])
    a = pd.concat([iv[KEY].reset_index(drop=True),a],axis=1)
    a["BETA_ALT"] = a.BETA; a["BETA"] *= orientation; influence *= orientation[None,:]
    a["sid"] = s["sid"]
    a.to_csv(OUT/"associations"/f"{s['sid']}_BMI.tsv",sep="\t",index=False)
    require(a.status.eq("ok").all(),f"Source {s['sid']} has invalid fixed IV association; no scenario-specific SNP dropping is allowed")
    source_assoc.append(a)
    ii = source_pool.get_indexer(ids)
    require((ii>=0).all(),"Source ID absent from source pool")
    M = sparse.csr_matrix((influence.ravel(),(np.repeat(ii,K),np.tile(np.arange(K),len(ii)))),shape=(len(source_pool),K))
    sparse_IF.append(M)
    strength_rows.append({"sid":s["sid"],"K_fixed":K,"mean_F_HC1":a.F_HC1.mean(),"min_F_HC1":a.F_HC1.min(),
                          "N_F_lt10":int(a.F_HC1.lt(10).sum()),"fraction_F_lt10":a.F_HC1.lt(10).mean(),
                          "N_source_sign_reversed_from_discovery":int(a.BETA.lt(0).sum()),"min_MAC":a.MAC.min()})
    del influence
source_strength = pd.DataFrame(strength_rows)
source_strength.to_csv(OUT/"qc/source_IV_strength.tsv",sep="\t",index=False)
IF_all = sparse.hstack(sparse_IF,format="csr")
SigmaX = (IF_all.T@IF_all).toarray()
require(np.allclose(np.sqrt(np.diag(SigmaX)),np.concatenate([a.SE_HC1 for a in source_assoc]),rtol=1e-9,atol=1e-12),"Covariance diagonal does not match source HC1 SE")
np.save(OUT/"associations/source_joint_cov.npy",SigmaX)
write_json(OUT/"associations/source_covariance_order.json",{"source_order":[s["sid"] for s in source_sets],"IV_order":iv.ID.tolist()})
source_overlap_IF = IF_all  # Retained until reporting for optional local checks.
del sparse_IF, G
gc.collect()
display(source_strength)

## 4. IVW / weighted median / MR-Egger 与配对差异

- **IVW：** 简单 first-order 权重、过原点回归，另存 multiplicative random-effects 的常见 NOME SE 和 Q heterogeneity。
- **Weighted median：** ratio 以 βX²/SEY² 加权，按加权分位点插值。有效性需要超过一半权重来自有效工具；不是对任意 pleiotropy 或 weak instruments 都稳健。
- **MR-Egger：** 加截距的加权回归，固定 discovery 定向；保存截距、Q、I²GX。需要 InSIDE 等假设，对 weak instruments/NOME 敏感。

方法定义参考开发者文档：[IVW](https://amymariemason.github.io/MR/reference/mr_ivw.html)、[weighted median](https://amymariemason.github.io/MR/reference/mr_median.html)、[MR-Egger](https://amymariemason.github.io/MR/reference/mr_egger.html)；[weighted median 原始论文](https://research-information.bris.ac.uk/ws/files/101501260/Bowden_et_al_2016_Genetic_Epidemiology.pdf)、[MR-Egger 原始论文](https://research-information.bris.ac.uk/ws/files/85723038/Int._J._Epidemiol._2015_Bowden_512_25.pdf)。不调用 R，这些点估计在 notebook 中明确实现。

主敏感性 CI 由联合 SNP 系数 Gaussian 重抽得到；同时传播 βX/βY 估计误差、残余 SNP correlation 及 source 重叠。MR 点估计本身仍是常用独立工具权重；联合重抽不修复弱工具偏差、pleiotropy 或高 LD。参数模型不为未知无效工具添加随机效应；常见 MRE SE 单独保留。

每个情境比较 `MR_scenario − MR_matched`，配对后再对重复取平均；重复不是独立实验。默认 2 个结局×4 个 mismatch 情境×3 方法，共 **24 个计划平均差异** 作 Bonferroni。若工具不足导致方法缺失，仍保留计划分母。P 是大样本正态近似，2000 次重抽只用来估计分布和 SE，不把超过次数计数当极小 P。

In [ ]:
def weighted_median(x,y,se_y):
    x,y,se_y = np.asarray(x,float),np.asarray(y,float),np.asarray(se_y,float)
    w = x*x/(se_y*se_y)
    ok = np.isfinite(w)&(w>0)&np.isfinite(x)&np.isfinite(y)
    if not ok.any(): return np.nan
    ratio = y[ok]/x[ok]; w = w[ok]
    order = np.argsort(ratio,kind="stable"); ratio = ratio[order]; w = w[order]
    positions = (np.cumsum(w)-.5*w)/w.sum()
    return float(np.interp(.5,positions,ratio))

def mr_points(x,y,se_y):
    """Batch estimates. Alleles stay aligned to discovery BMI-increasing alleles."""
    X = np.atleast_2d(x); Y = np.atleast_2d(y)
    X,Y = np.broadcast_arrays(X,Y)
    w = 1/np.asarray(se_y,float)**2
    den = np.sum(w*X*X,axis=1)
    ivw = np.divide(np.sum(w*X*Y,axis=1),den,out=np.full(len(X),np.nan),where=den>1e-20)
    median = np.full(len(X),np.nan); egger = np.full(len(X),np.nan)
    if X.shape[1]>=3:
        median = np.array([weighted_median(a,b,se_y) for a,b in zip(X,Y)])
        sw = w.sum(); xm = np.sum(w*X,axis=1)/sw; ym = np.sum(w*Y,axis=1)/sw
        d = np.sum(w*(X-xm[:,None])**2,axis=1)
        egger = np.divide(np.sum(w*(X-xm[:,None])*(Y-ym[:,None]),axis=1),d,
                          out=np.full(len(X),np.nan),where=d>1e-20)
    return {"IVW":ivw,"weighted_median":median,"MR_Egger":egger}

def mr_diagnostics(x,sx,y,sy):
    w = 1/sy**2; k = len(x); d = np.sum(w*x*x)
    theta = np.sum(w*x*y)/d
    Q = np.sum(w*(y-theta*x)**2)
    fese = 1/np.sqrt(d)
    rese = fese*np.sqrt(max(1,Q/(k-1))) if k>1 else fese
    out = {"IVW_Q":Q,"IVW_Q_df":k-1,"IVW_Q_P":stats.chi2.sf(Q,k-1) if k>1 else np.nan,
           "IVW_SE_FE_NOME":fese,"IVW_SE_MRE_NOME":rese,"max_IVW_weight_fraction":np.max(w*x*x)/d,
           "IVW_classic_CI_low":theta-1.959964*rese,"IVW_classic_CI_high":theta+1.959964*rese}
    if k>1:
        wx = 1/sx**2; bx = np.sum(wx*x)/wx.sum(); qx = np.sum(wx*(x-bx)**2)
        out["I2_GX"] = max(0,1-(k-1)/qx) if qx>0 else 0.
    else: out["I2_GX"] = np.nan
    if k>=3:
        sw = w.sum(); xm = np.sum(w*x)/sw; ym = np.sum(w*y)/sw
        dd = np.sum(w*(x-xm)**2)
        if dd>1e-20:
            slope = np.sum(w*(x-xm)*(y-ym))/dd; intercept = ym-slope*xm
            qe = np.sum(w*(y-intercept-slope*x)**2); scale = max(1,qe/(k-2))
            sei = np.sqrt(scale*(1/sw+xm*xm/dd)); ses = np.sqrt(scale/dd)
            out.update({"Egger_intercept":intercept,"Egger_intercept_SE_MRE_NOME":sei,
                        "Egger_intercept_P_normal":2*stats.norm.sf(abs(intercept/sei)),"Egger_SE_MRE_NOME":ses,
                        "Egger_Q":qe,"Egger_Q_df":k-2,"Egger_Q_P":stats.chi2.sf(qe,k-2)})
    return out

def covariance_root(V):
    V = (V+V.T)/2
    lam,U = linalg.eigh(V,check_finite=False)
    require(lam.min()>=-1e-7*max(lam.max(),1e-20),"Covariance has materially negative eigenvalues")
    return U*np.sqrt(np.maximum(lam,0))[None,:]

def uncertainty(point,draws):
    good = draws[np.isfinite(draws)]
    if not np.isfinite(point) or len(good)<.99*len(draws):
        return {"status":"too_few_IVs_or_unstable","SE_joint":np.nan,"CI_low":np.nan,"CI_high":np.nan,"P_joint_normal":np.nan,"N_boot_valid":len(good)}
    se = np.std(good,ddof=1)
    return {"status":"ok","SE_joint":se,"CI_low":point-1.959964*se,"CI_high":point+1.959964*se,
            "P_joint_normal":2*stats.norm.sf(abs(point/se)) if se>0 else np.nan,
            "bootstrap_percentile_low":np.quantile(good,.025),"bootstrap_percentile_high":np.quantile(good,.975),
            "bootstrap_median_minus_point":np.median(good)-point,"N_boot_valid":len(good)}

Xhat = np.array([a.BETA.to_numpy() for a in source_assoc])
Xse = np.array([a.SE_HC1.to_numpy() for a in source_assoc])
rng = stable_rng("joint-coefficient-resampling")
# One joint X draw preserves overlap within/across source groups and replicates.
drawX = (Xhat.ravel()[None,:]+rng.standard_normal((N_BOOT,S*K))@covariance_root(SigmaX).T).reshape(N_BOOT,S,K)
mr_rows, contrast_rows, mean_rows, leave_one_out = [], [], [], []
all_boot, all_points = {}, {}
for outcome in OUTCOMES:
    y = target_assoc[outcome].BETA.to_numpy(); sy = target_assoc[outcome].SE_HC1.to_numpy()
    # Identical target Y draw reused for every source, never regenerated per scenario.
    drawY = y[None,:]+rng.standard_normal((N_BOOT,K))@covariance_root(target_cov[outcome]).T
    boots = {method:np.empty((N_BOOT,S)) for method in METHODS}
    points = {method:np.empty(S) for method in METHODS}
    for j,s in enumerate(source_sets):
        p = mr_points(Xhat[j],y,sy); b = mr_points(drawX[:,j,:],drawY,sy)
        diag = mr_diagnostics(Xhat[j],Xse[j],y,sy)
        for method in METHODS:
            estimate = float(p[method][0]); boots[method][:,j] = b[method]; points[method][j] = estimate
            mr_rows.append({"outcome":outcome,"sid":s["sid"],"repeat":s["repeat"],"scenario":s["scenario"],"method":method,
                            "estimate":estimate,"K_fixed":K,**uncertainty(estimate,b[method]),**diag})
        if K>1:
            for drop in range(K):
                keep = np.arange(K)!=drop
                theta = np.sum(Xhat[j,keep]*y[keep]/sy[keep]**2)/np.sum(Xhat[j,keep]**2/sy[keep]**2)
                leave_one_out.append({"outcome":outcome,"sid":s["sid"],"dropped_IV":iv.ID.iloc[drop],"IVW_leave_one_out":theta})
    all_boot[outcome],all_points[outcome] = boots,points
    np.savez_compressed(OUT/"MR"/f"{outcome}_joint_MR_draws.npz",**boots)
    for name,_ in SCENARIOS:
        jj = [j for j,s in enumerate(source_sets) if s["scenario"]==name]
        for method in METHODS:
            point = float(np.mean(points[method][jj])); b = np.mean(boots[method][:,jj],axis=1)
            mean_rows.append({"outcome":outcome,"scenario":name,"method":method,"estimate_mean_repeats":point,
                              "K_fixed":K,**uncertainty(point,b)})
    for name,_ in SCENARIOS:
        if name=="matched": continue
        for method in METHODS:
            ds,db = [],[]
            for rep in range(1,N_REPEATS+1):
                j = next(j for j,s in enumerate(source_sets) if s["repeat"]==rep and s["scenario"]==name)
                m = next(j for j,s in enumerate(source_sets) if s["repeat"]==rep and s["scenario"]=="matched")
                delta = points[method][j]-points[method][m]
                bd = boots[method][:,j]-boots[method][:,m]
                contrast_rows.append({"outcome":outcome,"scenario":name,"method":method,"repeat":rep,
                                      "contrast":"scenario minus matched","delta":delta,**uncertainty(delta,bd)})
                ds.append(delta); db.append(bd)
            delta = float(np.mean(ds)); bd = np.mean(db,axis=0)
            contrast_rows.append({"outcome":outcome,"scenario":name,"method":method,"repeat":"mean",
                                  "contrast":"scenario minus matched","delta":delta,**uncertainty(delta,bd)})
mr = pd.DataFrame(mr_rows).merge(sample_qc,on=["sid","repeat","scenario"],validate="many_to_one").merge(source_strength,on="sid",validate="many_to_one",suffixes=("","_strength"))
contrasts = pd.DataFrame(contrast_rows)
contrasts["P_Bonf_planned_contrasts"] = np.where(contrasts.repeat.eq("mean"),np.minimum(contrasts.P_joint_normal*CONTRAST_FAMILY,1),np.nan)
mean_mr = pd.DataFrame(mean_rows)
mr.to_csv(OUT/"MR/MR_all_sources.tsv",sep="\t",index=False)
mean_mr.to_csv(OUT/"MR/MR_mean_repeats.tsv",sep="\t",index=False)
contrasts.to_csv(OUT/"MR/paired_age_mismatch_contrasts.tsv",sep="\t",index=False)
pd.DataFrame(leave_one_out).to_csv(OUT/"MR/IVW_leave_one_out.tsv",sep="\t",index=False)
display(mean_mr)
display(contrasts.loc[contrasts.repeat.eq("mean")])

## 5. 图、输出和解释规则

首先看 `design/age_mixture_QC.tsv` 和 `qc/source_IV_strength.tsv`；再看 `MR/MR_mean_repeats.tsv` 的效应轨迹，以及 **`MR/paired_age_mismatch_contrasts.tsv` 中 repeat=mean 的配对差异、CI 和校正 P**。IVW leave-one-out、Q、Egger 截距、I²GX 和最大 IV 权重用于检查某个 SNP 或弱工具是否主导。

若 source 年龄偏移增大时，MR 相对 matched 出现一致、可重复的漂移，且工具强度和单点诊断不能简单解释，可支持“naive MR 对来源年龄构成敏感”。robust MR 同向变化也可能发生：当许多工具的 βX 一起改变时，median/Egger 不会自动将 βX 转换到目标年龄分布。

matched 是对齐参考，不是真实因果效应。观察到变化不能单独证明 age-mismatch bias；年龄相关 cohort/composition、效应异质性、pleiotropy 和模型误设仍需区分。未观察到变化也不证明不存在问题，尤其是 discovery 选出的工具很少、分割后 F 低、CI 很宽时。这是候选面板、固定 IV 的 baseline，尚未比较 IPW/regression/DR/TLMR。

所有角色 keep、ALT genotype 和 influence 中间信息均留在当前 workspace，不需要导出个体数据。只分享允许披露的聚合表和图。`ANALYSIS_COMPLETE.json` 只在全部步骤成功后写入。

**本地核查范围：** notebook 结构和语法、合成 PGEN 的完整执行、PLINK pooled GWAS/clump/ALT export、独立 statsmodels 对照的 OLS/HC1/IVW/Egger、加权 median 已知例子、重叠 covariance 与配对重抽。合成结果不代表真实 AoU B3 结果；本地未运行 controlled cohort。

In [ ]:
scenario_names = [name for name,_ in sorted(SCENARIOS,key=lambda z:z[1])]
labels = {"younger_strong":"Younger (strong)","younger_moderate":"Younger (moderate)",
          "matched":"Age matched","older_moderate":"Older (moderate)","older_strong":"Older (strong)"}
colors = {"IVW":"#245b9e","weighted_median":"#14866d","MR_Egger":"#a24582"}

fig,ax = plt.subplots(figsize=(8,4))
for name in scenario_names:
    a = sample_qc.loc[sample_qc.scenario.eq(name)]
    ax.scatter(a.age_mean,a.age_SD,label=labels[name],s=45)
ax.axvline(target_age.mean(),color="black",ls="--",lw=1,label="Fixed BP target mean age")
ax.set(xlabel="BMI source mean age (years)",ylabel="BMI source age SD (years)",title=f"Age-mixture sampling; N = {N_SOURCE:,} per source")
ax.legend(fontsize=8,bbox_to_anchor=(1.01,1),loc="upper left")
fig.tight_layout(); fig.savefig(OUT/"figures/age_mixture_QC.png",dpi=180,bbox_inches="tight"); plt.show()

for outcome in OUTCOMES:
    # Separate axes prevent a very imprecise Egger estimate hiding the IVW scale.
    fig,axes = plt.subplots(1,3,figsize=(13,4.3))
    for ax,method in zip(axes,METHODS):
        a = mean_mr.loc[mean_mr.outcome.eq(outcome)&mean_mr.method.eq(method)].set_index("scenario").reindex(scenario_names)
        good = a.status.eq("ok")
        yy = np.arange(len(a))
        ax.errorbar(a.loc[good,"estimate_mean_repeats"],yy[good],xerr=1.959964*a.loc[good,"SE_joint"],fmt="o",color=colors[method],capsize=3)
        ax.set_yticks(yy,[labels[name] for name in scenario_names] if ax is axes[0] else [""]*len(yy))
        ax.axvline(0,color="grey",lw=.7); ax.invert_yaxis()
        ax.set(title=method,xlabel="mmHg per BMI unit")
        if not good.all(): ax.text(.02,.02,"Some estimates unavailable",transform=ax.transAxes,fontsize=8)
    fig.suptitle(f"BMI → {outcome}: fixed IVs and target; mean over paired source repeats\n95% joint coefficient-resampling CIs (conditional sensitivity analysis)",fontsize=11)
    fig.tight_layout(); fig.savefig(OUT/"figures"/f"MR_{outcome}.png",dpi=180,bbox_inches="tight"); plt.show()
    fig,axes = plt.subplots(1,3,figsize=(13,4.3))
    names = [name for name in scenario_names if name!="matched"]
    for ax,method in zip(axes,METHODS):
        a = contrasts.loc[contrasts.outcome.eq(outcome)&contrasts.method.eq(method)&contrasts.repeat.eq("mean")].set_index("scenario").reindex(names)
        good = a.status.eq("ok"); yy = np.arange(len(a))
        ax.errorbar(a.loc[good,"delta"],yy[good],xerr=1.959964*a.loc[good,"SE_joint"],fmt="o",color=colors[method],capsize=3)
        ax.set_yticks(yy,[labels[name] for name in names] if ax is axes[0] else [""]*len(yy))
        ax.axvline(0,color="black",lw=1); ax.invert_yaxis()
        ax.set(title=method,xlabel="Difference from age-matched MR")
    fig.suptitle(f"{outcome}: paired age-mismatch contrasts\nShared target and source overlap included in joint resampling",fontsize=11)
    fig.tight_layout(); fig.savefig(OUT/"figures"/f"paired_contrasts_{outcome}.png",dpi=180,bbox_inches="tight"); plt.show()

write_json(OUT/"ANALYSIS_COMPLETE.json",{
    "status":"complete","finished_utc":datetime.now(timezone.utc).isoformat(),"design":design,
    "linear_run":str(LINEAR_RUN),"plink_version":PLINK_VERSION,"K_fixed":K,"N_MR_rows":len(mr),
    "N_mean_contrasts":int(contrasts.repeat.eq("mean").sum()),
    "inference":"HC1 coefficient covariance, shared-target and overlapping-source joint Gaussian resampling; fixed discovery IVs",
    "interpretation":"candidate-panel MR age-mixture sensitivity; not causal truth or longitudinal aging proof"})
print("B3 completed:",OUT)
print("Main tables: MR/MR_mean_repeats.tsv and MR/paired_age_mismatch_contrasts.tsv")